# genpcb — 準備 SFT 種子資料（D-sft v0）

產出生成器 SFT 的種子資料：程序化板 → compact placement DSL → jsonl。
**不需要 GPU**（純 CPU + 下載 tokenizer 做長度驗證）。

紀律（docs/generator-model-selection.md §3.5）：邏輯一律在 `src/genpcb/data/`，
本 notebook 只 orchestrate + 視覺化。

範圍提醒：這是**生成器 SFT** 種子資料。檢查器（Model A）需要的 Freerouting
標註資料是另一條 CPU farm 管線（需 KiCad+Java），不在本 notebook、不在 Colab。
v0 板為合理種子、非最佳化擺位（見 docs/data-engine.md §1.1）。

In [ ]:
# ── 參數 ──
MODEL_CONFIG = 'model_gemma4_12b'   # tokenizer 長度驗證對象
N_BOARDS = 3000
GRID_MM = 0.1
ON_COLAB = True                     # 本機跑改 False

In [ ]:
# ── (Colab) 拉程式碼 + secrets + Drive ──
if ON_COLAB:
    import os, os.path
    from google.colab import drive, userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # 下載 Gemma tokenizer（gated）
    REPO_AUTH = f"https://{userdata.get('GH_TOKEN')}@github.com/timmytsaa/genpcb.git"
    if os.path.isdir('/content/genpcb'):
        !git -C /content/genpcb pull
    else:
        !git clone {REPO_AUTH} /content/genpcb
    %pip install -q -e /content/genpcb
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/genpcb/data/sft/boards.jsonl'   # Drive 持久化
else:
    OUT = 'data/sft/boards.jsonl'

In [ ]:
# ── 產生 + 序列化 + 寫 jsonl（邏輯在 src）──
from genpcb.data.build import build
rows = build(N_BOARDS, OUT, grid=GRID_MM)
chars = [r['chars'] for r in rows]
print(f'{len(rows)} boards → {OUT}')
print(f'chars/board: mean {sum(chars)//len(chars)}  max {max(chars)}')
from collections import Counter
print('家族分布:', dict(Counter(r['family'] for r in rows)))

In [ ]:
# ── token 長度驗證（對 MODEL_CONFIG 的 tokenizer）──
from genpcb.config import load_config
from genpcb.models.adapter import ModelAdapter
adapter = ModelAdapter(load_config(f'/content/genpcb/configs/{MODEL_CONFIG}.yaml' if ON_COLAB else f'configs/{MODEL_CONFIG}.yaml'))
tok = adapter.load_tokenizer()
toks = [len(tok.encode(r['text'], add_special_tokens=False)) for r in rows]
ctx = adapter.spec.max_seq_len
import numpy as np
a = np.array(toks)
print(f'model: {adapter.spec.model_id}  (context {ctx})')
print(f'tokens/board: mean {a.mean():.0f}  p50 {np.percentile(a,50):.0f}  '
      f'p95 {np.percentile(a,95):.0f}  max {a.max()}')
print(f'超過 context: {(a>ctx).sum()}/{len(a)}')

In [ ]:
# ── 視覺化：token 長度分布（notebook 只消費）──
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7,3))
for fam in sorted(set(r['family'] for r in rows)):
    fam_toks = [t for t,r in zip(toks,rows) if r['family']==fam]
    ax.hist(fam_toks, bins=40, alpha=0.6, label=fam)
ax.axvline(ctx, color='r', ls='--', label=f'context {ctx}')
ax.set_xlabel('tokens/board'); ax.set_ylabel('count'); ax.legend(); ax.set_title('SFT 種子板 token 長度')
plt.tight_layout(); plt.show()

In [ ]:
# ── 抽檢一筆樣本 ──
print(rows[0]['family'], '| comps', rows[0]['n_components'], '| nets', rows[0]['n_nets'])
print(rows[0]['text'])

## 接下來

- 種子資料已落 `OUT`（Colab 上是 Drive）。`genpcb.train.sft` 的 config 指向
  `data/sft/boards.jsonl`——本機訓練可直接用；Colab 訓練從 Drive 讀。
- 品質升級（非本 notebook）：placement 合成器（SA/force-directed）、真實板
  Stream R、ERC-clean 模板文法（docs/data-engine.md §1）。
- 格式 v0 僅含 placement（形態 A）。GRPO reward 需要的 `dsl → .kicad_pcb`
  還原器是 Phase 0 工件（docs/output-format.md）。